<a href="https://colab.research.google.com/github/michaelsteven1299/proyecto_michael-/blob/main/src/01_consolidar.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Universidad Libre - Seccional Cali<br>Facultad de Ingeniería - Diplomado en Ciencia de Datos<br>(ↄ) Diego Fernando Marin, 2024

# 01_consolidar
Plantilla para el desarrollo del proyecto del diplomado de Ciencia de Datos, aplicando buenas prácticas.

---

Este cuaderno se enfoca en la integración de las distintas fuentes de datos en un formato cohesivo y estructurado. Aquí transformamos múltiples conjuntos de datos en una base unificada que servirá para los análisis posteriores.

**Propósito:** Crear una vista unificada y coherente de todos los datos recolectados, facilitando su posterior procesamiento y análisis.

**Tareas habituales:**
- Renombrar archivos
- Unión vertical de archivos complementarios (`union`)
- Combinar archivos (`joins`: inner, left, right, full outer)
- Estandarización inicial de formatos de columnas
- Verificación de consistencia en las uniones
- Validación de cardinalidad en las relaciones
- Gestión de duplicados producto de las uniones

In [9]:
from google.colab import drive
import os, pandas as pd

drive.mount('/content/drive')

RAW_PATH     = "/content/drive/MyDrive/proyecto_oro/data/raw/"
LANDING_PATH = "/content/drive/MyDrive/proyecto_oro/data/landing/"
os.makedirs(LANDING_PATH, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [10]:
# Recorremos todos los archivos de raw/ y les ponemos un prefijo con
# el nombre de la variable, para que no se mezclen columnas iguales
# (ej. "Close") de distintas fuentes.
dfs = []

for archivo in os.listdir(RAW_PATH):
    if not archivo.endswith('.csv'):
        continue

    nombre = archivo.replace('.csv', '')
    df = pd.read_csv(RAW_PATH + archivo, index_col=0, parse_dates=True)

    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)

    df.columns = [f"{nombre}_{col}" for col in df.columns]
    dfs.append(df)
    print(f"✓ {archivo}: {df.shape[0]:,} filas × {df.shape[1]} columnas")

✓ dxy.csv: 1,380 filas × 5 columnas
✓ usd_cop.csv: 1,428 filas × 5 columnas
✓ wti_crudo.csv: 1,380 filas × 5 columnas
✓ vix.csv: 1,379 filas × 5 columnas
✓ oro_xauusd.csv: 1,378 filas × 5 columnas
✓ trm.csv: 1,299 filas × 1 columnas
✓ bono_10y.csv: 1,378 filas × 5 columnas


In [11]:
# join='outer' conserva TODAS las fechas de TODAS las fuentes, aunque
# algunas queden con NaN donde esa fuente no tenía dato ese día.
# La decisión de qué hacer con esos NaN se toma en el cuaderno de
# Limpieza, no aquí.
df_landing = pd.concat(dfs, axis=1, join='outer')
df_landing = df_landing.sort_index()

# Nos quedamos solo con días hábiles bancarios (lunes a viernes),
# ya que la TRM y varios mercados no operan fines de semana
df_landing = df_landing[df_landing.index.dayofweek < 5]

print(f"Filas después de unir y filtrar fines de semana: {len(df_landing):,}")

Filas después de unir y filtrar fines de semana: 1,431


In [12]:
# oro_cop: precio del oro convertido a pesos colombianos, usando la
# TRM oficial (certificada por la Superfinanciera) como tasa de cambio,
# no el precio de mercado (usd_cop), para mayor rigor metodológico
df_landing['oro_cop'] = df_landing['oro_xauusd_Close'] * df_landing['trm_trm']

In [13]:
df_landing.index.name = 'DATE'
df_landing = df_landing.reset_index()
df_landing['DATE'] = df_landing['DATE'].astype(str)

ruta_salida = LANDING_PATH + "consolidado_oro_dxy.csv"
df_landing.to_csv(ruta_salida, index=False)

print(f"\n✅ Landing listo (con NaN sin tratar — eso se resuelve en Limpieza)")
print(f"📊 {df_landing.shape[0]:,} filas × {df_landing.shape[1]} columnas")
print(f"📅 {df_landing['DATE'].min()} → {df_landing['DATE'].max()}")
df_landing.head()


✅ Landing listo (con NaN sin tratar — eso se resuelve en Limpieza)
📊 1,431 filas × 33 columnas
📅 2021-01-01 → 2026-07-01


,DATE,dxy_Close,dxy_High,dxy_Low,dxy_Open,dxy_Volume,usd_cop_Close,usd_cop_High,usd_cop_Low,usd_cop_Open,...,oro_xauusd_Low,oro_xauusd_Open,oro_xauusd_Volume,trm_trm,bono_10y_Close,bono_10y_High,bono_10y_Low,bono_10y_Open,bono_10y_Volume,oro_cop
0,2021-01-01,NaN,NaN,NaN,NaN,NaN,3420.250000,3420.250000,3420.250000,3420.250000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2021-01-04,89.879997,89.940002,89.419998,89.930000,0.0,3420.250000,3438.090088,3381.986816,3420.250000,...,180.960007,181.970001,14331400.0,NaN,0.917,0.953,0.907,0.935,0.0,NaN
2,2021-01-05,89.440002,89.900002,89.430000,89.900002,0.0,3447.750000,3467.550049,3428.679932,3447.750000,...,181.820007,182.869995,12718800.0,3420.78,0.955,0.963,0.927,0.937,0.0,625558.021897
3,2021-01-06,89.529999,89.800003,89.209999,89.480003,0.0,3442.250000,3442.250000,3401.046387,3442.250000,...,178.240005,181.490005,18453500.0,3450.74,1.042,1.054,1.000,1.000,0.0,620788.104938
4,2021-01-07,89.830002,89.970001,89.320000,89.320000,0.0,3412.800049,3477.570068,3374.599121,3412.800049,...,178.839996,179.690002,7110200.0,3428.04,1.071,1.088,1.054,1.056,0.0,615264.604554
